# Unfallatlas Deutschland — U-Phase

**Phase:** Understanding the Data (U) · 2 of 5 · QUA³CK
**Goal of this notebook:** verify the Q-phase assumptions against the data,
audit the dataset for quality and leakage, characterise its distributions
and patterns, and produce a column-by-column preprocessing decision table
that A³ can implement against.

**Strict scope.** This notebook *observes*, *audits*, and *decides*. It does
not encode, scale, engineer features, or train models. All transformations
are A³ work; their selection rationale is documented here.

> Every claim in this notebook must be backed by a cell output above it.
> Every preprocessing decision in this notebook must appear in the §10
> decision table as a contract to A³.

---

## Position in the QUA³CK process

| Phase | Notebook | Status |
|:---|:---|:---:|
| Q — Question | `01_Q_Phase.ipynb` | ✓ |
| **U — Understanding** | `02_U_Phase.ipynb` | **→ here** |
| A³ — Algorithm / Adapt / Adjust | `03_A3_Phase.ipynb` | pending |
| C — Conclude & Compare | `04_C_Phase.ipynb` | pending |
| K — Knowledge Transfer | `app/streamlit_app.py` | pending |

---

## 0 — Setup and reproducibility

A U-phase notebook is reproducible or it is not a U-phase notebook. The
following cells pin versions, hash the data file, configure Plotly, and set
a deterministic random seed for sampling.

**Plotly is the only plotting library used here** — it is already pinned in
`pyproject.toml` (≥ 5.22) and renders interactively in Jupyter and VS Code.
Figures save as standalone `.html` files (preserve interactivity) and
optionally as `.png` if `kaleido` is installed (`uv pip install kaleido`).

In [ ]:
# Standard library
import hashlib
import subprocess
from collections import Counter
from datetime import datetime
from math import log2
from pathlib import Path

# Third-party — all in pyproject.toml
import duckdb
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

# Plotly defaults — clean white template, sensible figure size, mobile-friendly
pio.templates.default = "plotly_white"
pio.renderers.default = "vscode"
DEFAULT_FIG_W, DEFAULT_FIG_H = 900, 480

# Pandas display
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")
np.random.seed(42)

# Project colour palette — kept consistent across all U-phase plots
COLOR_PRIMARY = "#3a6ea5"   # neutral blue (default categorical)
COLOR_FATAL   = "#c1393a"   # class 1 — Getötet
COLOR_SERIOUS = "#e09f3e"   # class 2 — Schwer
COLOR_MINOR   = "#5a8db8"   # class 3 — Leicht
COLOURS_SEV   = [COLOR_FATAL, COLOR_SERIOUS, COLOR_MINOR]  # ordered 1, 2, 3


In [ ]:
# Paths — robust whether launched from project root or from notebooks/
BASE_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA     = BASE_DIR / "data" / "body.parquet"
FIG_DIR  = BASE_DIR / "reports" / "figures" / "u_phase"
FIG_DIR.mkdir(parents=True, exist_ok=True)

assert DATA.exists(), (
    f"Data file not found at {DATA}\n"
    "Re-run the consolidation script in src/unfallatlas/data/ to materialise it."
)

def save_fig(fig: go.Figure, slug: str) -> tuple[Path, Path | None]:
    """Persist a Plotly figure as interactive HTML and optionally as static PNG.

    HTML always saves; PNG requires kaleido (uv pip install kaleido).
    Returns (html_path, png_path_or_None).
    """
    html_path = FIG_DIR / f"{slug}.html"
    fig.write_html(html_path, include_plotlyjs="cdn", full_html=True)
    png_path = None
    try:
        png_path = FIG_DIR / f"{slug}.png"
        fig.write_image(png_path, scale=2, width=DEFAULT_FIG_W, height=DEFAULT_FIG_H)
    except (ValueError, ImportError, Exception):
        png_path = None
    return html_path, png_path


In [ ]:
# Provenance — recorded once, at the top, for the run.
def _git_short_sha():
    try:
        return subprocess.check_output(
            ["git", "rev-parse", "--short", "HEAD"], stderr=subprocess.DEVNULL
        ).decode().strip()
    except Exception:
        return "unknown"

import plotly
provenance = {
    "source":      "GovData / Mobilithek — Unfallatlas Deutschland",
    "licence":     "Datenlizenz Deutschland 2.0 (Namensnennung)",
    "file":        str(DATA.relative_to(BASE_DIR)),
    "size_mb":     round(DATA.stat().st_size / 1_048_576, 2),
    "sha256_16":   hashlib.sha256(DATA.read_bytes()).hexdigest()[:16],
    "duckdb":      duckdb.__version__,
    "pandas":      pd.__version__,
    "numpy":       np.__version__,
    "plotly":      plotly.__version__,
    "git_commit":  _git_short_sha(),
    "run_at_utc":  datetime.utcnow().isoformat(timespec="seconds") + "Z",
    "random_seed": 42,
}
for k, v in provenance.items():
    print(f"  {k:14s} {v}")


In [ ]:
# DuckDB connection — used for any aggregation over the full ~2 M-row table.
con = duckdb.connect()
con.execute("SET memory_limit = '4GB';")


## 1 — Schema and dtype audit

The first quantitative output: what columns exist, what are their dtypes, and
how many distinct values does each one carry?

In [ ]:
schema = con.execute(f"DESCRIBE SELECT * FROM '{DATA}'").df()
print(f"columns: {len(schema)}")
schema


In [ ]:
# Cardinality + missingness — the audit one-pager.
def audit_table(parquet_path: Path) -> pd.DataFrame:
    cols = con.execute(f"DESCRIBE SELECT * FROM '{parquet_path}'").df()["column_name"]
    rows = []
    for col in cols:
        q = f"""
            SELECT
                COUNT(*) AS n,
                COUNT(DISTINCT "{col}") AS n_unique,
                SUM(CASE WHEN "{col}" IS NULL THEN 1 ELSE 0 END) AS n_missing
            FROM '{parquet_path}'
        """
        r = con.execute(q).fetchone()
        rows.append({
            "column": col,
            "n_unique": r[1],
            "n_missing": r[2],
            "pct_missing": 100.0 * r[2] / r[0],
        })
    return pd.DataFrame(rows)

audit = audit_table(DATA)
audit


### Semantic-type annotation

Storage dtype is not modelling type. The table below records, for each column,
how A³ should treat it. This annotation is the most useful single artefact U
hands to A³.

| Column | Storage | Semantic type | Treatment hint for A³ |
|:---|:---|:---|:---|
| `OBJECTID` | INT | identifier | drop before modelling |
| `UJAHR` | SMALLINT | temporal ordinal | only used for the chronological split, not as a feature |
| `UMONAT` | TINYINT | cyclic ordinal (period 12) | sin/cos encoding in A³ |
| `USTUNDE` | TINYINT | cyclic ordinal (period 24) | sin/cos encoding in A³ |
| `UWOCHENTAG` | TINYINT | cyclic ordinal (period 7) | sin/cos encoding in A³ |
| **`UKATGEORIE`** | TINYINT | **ordinal target** | label — never a feature |
| `UART` | TINYINT | nominal categorical | one-hot or target-encoded; **leakage probe required** |
| `UTYP1` | TINYINT | nominal categorical | one-hot or target-encoded; **leakage probe required** |
| `ULICHTVERH` | TINYINT | nominal categorical (3 levels) | one-hot |
| `STRZUSTAND` | TINYINT | nominal categorical (3 levels) | one-hot |
| `IstRad` … `IstSonstig` | BOOLEAN | binary | pass-through |
| `LON`, `LAT` | DOUBLE | continuous spatial | `StandardScaler` only if a distance-based model is added; tree models pass-through |
| `UREGBEZ` | VARCHAR | nominal categorical | target-encoding |
| `UKREIS` | VARCHAR | high-cardinality nominal | target-encoding; one-hot would explode the matrix |
| `UGEMEINDE` | VARCHAR | very-high-cardinality nominal | drop or hash-encode |

> **Note.** `UART` and `UTYP1` are flagged for leakage probing in §9 because
> their semantic relationship to the severity outcome is not obvious from
> the documentation alone.

---

## 2 — Volume and temporal coverage

Verifies the Q-phase assumption: ~2.09 M rows, 9 vintages 2016 – 2024.

In [ ]:
total = con.execute(f"""
    SELECT COUNT(*) AS n_rows,
           COUNT(DISTINCT UJAHR) AS n_years,
           MIN(UJAHR) AS first_year,
           MAX(UJAHR) AS last_year
    FROM '{DATA}'
""").df()
total


In [ ]:
by_year = con.execute(f"""
    SELECT UJAHR AS year,
           COUNT(*) AS n_rows,
           ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct
    FROM '{DATA}'
    GROUP BY UJAHR
    ORDER BY UJAHR
""").df()
by_year


In [ ]:
fig = px.bar(
    by_year, x="year", y="n_rows", text="n_rows",
    title="Accidents per year — Unfallatlas 2016 – 2024",
    labels={"year": "", "n_rows": "accidents"},
    color_discrete_sequence=[COLOR_PRIMARY],
    height=DEFAULT_FIG_H,
)
fig.update_traces(
    texttemplate="%{text:,}", textposition="outside",
    hovertemplate="<b>%{x}</b><br>%{y:,} accidents<extra></extra>",
)
fig.update_layout(xaxis=dict(tickmode="linear"), showlegend=False)
save_fig(fig, "02_rows_per_year")
fig.show()


> **Observation.** All nine years are present. The visible 2020 dip is
> consistent with COVID-19 mobility reduction and matches BASt yearly
> reports — structural, not a data-quality issue. The level recovers from
> 2021 onwards but does not return to the 2016 – 2019 baseline. A³ should
> note that the 2024 test year sits on a slightly different volume regime
> than the 2016 – 2019 portion of training.

---

## 3 — Data quality audit

Five sub-audits: missingness, sentinel values, duplicates, range checks,
and consistency rules.

### 3.1  Missing values

In [ ]:
missing = audit[audit["n_missing"] > 0].sort_values("pct_missing", ascending=False)
print(f"columns with missing values: {len(missing)} / {len(audit)}")
missing


In [ ]:
# Missingness map — fully native Plotly (no missingno dependency).
SAMPLE_N = 20_000
df_sample = con.execute(
    f"SELECT * FROM '{DATA}' USING SAMPLE {SAMPLE_N} ROWS (reservoir, 42)"
).df()

miss_matrix = df_sample.isna().astype(int).T  # rows = columns, cols = sample rows
fig = px.imshow(
    miss_matrix.values,
    aspect="auto",
    color_continuous_scale=[[0, "#f4f4f4"], [1, "#1a1a1a"]],
    y=miss_matrix.index.tolist(),
    labels={"x": "row index in sample", "y": "column", "color": "missing"},
    title=f"Missingness map — dark = NaN (n = {SAMPLE_N:,} sampled rows)",
    height=520,
)
fig.update_xaxes(showticklabels=False)
fig.update_coloraxes(showscale=False)
save_fig(fig, "03_missingness_matrix")
fig.show()


> **Decision.** Per-column missing-value strategies are recorded in the §10
> decision table. Imputation is *not* performed here — that is A³ work
> inside a `Pipeline` so statistics do not leak across splits.

### 3.2  Sentinel-value scan

Sentinels (e.g. `-1`, `9999`, empty strings) are not caught by `isna()` and
must be hunted explicitly. The Unfallatlas codebook uses small positive
integers, so any negative numeric value is suspicious.

In [ ]:
# Min / max of every numeric column.
numeric_cols = ["UJAHR", "UMONAT", "USTUNDE", "UWOCHENTAG", "UKATGEORIE",
                "UART", "UTYP1", "ULICHTVERH", "STRZUSTAND", "LON", "LAT"]
minmax = con.execute(f"""
    SELECT {', '.join(f'MIN({c}) AS min_{c}, MAX({c}) AS max_{c}' for c in numeric_cols)}
    FROM '{DATA}'
""").df().T
minmax.columns = ["value"]
minmax


In [ ]:
# String-column sentinel-like values.
str_cols = ["UREGBEZ", "UKREIS", "UGEMEINDE"]
suspicious_set = {"unknown", "na", "n/a", "", "null", "none", "?", "-"}
for col in str_cols:
    df_col = con.execute(
        f"SELECT DISTINCT \"{col}\" AS v FROM '{DATA}'"
    ).df()["v"].astype(str)
    suspicious = df_col[df_col.str.lower().isin(suspicious_set)]
    print(f"  {col:10s}  distinct = {len(df_col):>6}  "
          f"suspicious = {len(suspicious)}")


### 3.3  Duplicates

In [ ]:
dupe_ids = con.execute(f"""
    SELECT OBJECTID, COUNT(*) AS n
    FROM '{DATA}'
    GROUP BY OBJECTID
    HAVING COUNT(*) > 1
    LIMIT 10
""").df()
print(f"duplicate OBJECTIDs: {len(dupe_ids):,}")
if len(dupe_ids):
    display(dupe_ids)


In [ ]:
# Exact row duplicates across all 21 columns.
n_total    = con.execute(f"SELECT COUNT(*) FROM '{DATA}'").fetchone()[0]
n_distinct = con.execute(f"SELECT COUNT(*) FROM (SELECT DISTINCT * FROM '{DATA}')").fetchone()[0]
print(f"  total rows          : {n_total:,}")
print(f"  distinct rows       : {n_distinct:,}")
print(f"  exact row duplicates: {n_total - n_distinct:,}")


### 3.4  Range checks

Domain-bound checks catch errors that statistical outlier rules will miss.

In [ ]:
DE_BBOX = {"lat_min": 47.27, "lat_max": 55.06, "lon_min": 5.87, "lon_max": 15.04}

range_checks = con.execute(f"""
    SELECT
        SUM(CASE WHEN LAT NOT BETWEEN {DE_BBOX['lat_min']} AND {DE_BBOX['lat_max']}
                 THEN 1 ELSE 0 END) AS lat_outside_de,
        SUM(CASE WHEN LON NOT BETWEEN {DE_BBOX['lon_min']} AND {DE_BBOX['lon_max']}
                 THEN 1 ELSE 0 END) AS lon_outside_de,
        SUM(CASE WHEN USTUNDE    NOT BETWEEN 0 AND 23 THEN 1 ELSE 0 END) AS hour_invalid,
        SUM(CASE WHEN UMONAT     NOT BETWEEN 1 AND 12 THEN 1 ELSE 0 END) AS month_invalid,
        SUM(CASE WHEN UWOCHENTAG NOT BETWEEN 1 AND 7  THEN 1 ELSE 0 END) AS weekday_invalid,
        SUM(CASE WHEN UKATGEORIE NOT IN (1, 2, 3)     THEN 1 ELSE 0 END) AS severity_invalid
    FROM '{DATA}'
""").df().T
range_checks.columns = ["count"]
range_checks


### 3.5  Consistency rules

Domain rule: every accident must involve at least one transport mode.

In [ ]:
mode_cols = ["IstRad", "IstPKW", "IstFuss", "IstKrad", "IstGkfz", "IstSonstig"]
violation = con.execute(f"""
    SELECT COUNT(*) AS n_no_mode
    FROM '{DATA}'
    WHERE CAST(IstRad AS INT) + CAST(IstPKW AS INT) + CAST(IstFuss AS INT)
        + CAST(IstKrad AS INT) + CAST(IstGkfz AS INT) + CAST(IstSonstig AS INT) = 0
""").df()
print("rows with no transport mode flagged:", int(violation.iloc[0, 0]))


> **Quality audit summary.** Document each finding above in the §11 risk
> list if it affects A³. Range failures and consistency violations above
> 0.1 % of rows warrant a return to Q for re-discussion.

---

## 4 — Target variable

Verifies the Q-phase assumption of class distribution ≈ 1 / 18 / 81.

In [ ]:
target = con.execute(f"""
    SELECT UKATGEORIE AS class,
           CASE UKATGEORIE WHEN 1 THEN '1 — Getötet'
                          WHEN 2 THEN '2 — Schwer verletzt'
                          ELSE     '3 — Leicht verletzt' END AS label,
           COUNT(*) AS n,
           ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct
    FROM '{DATA}'
    GROUP BY UKATGEORIE
    ORDER BY UKATGEORIE
""").df()
target


In [ ]:
n_max = target["n"].max()
n_min = target["n"].min()
print(f"imbalance ratio (majority : minority) = {n_max / n_min:.1f} : 1")


In [ ]:
# Horizontal stacked bar — single bar, three segments, percentages annotated.
fig = go.Figure()
left = 0
for (_, row), color in zip(target.iterrows(), COLOURS_SEV):
    fig.add_trace(go.Bar(
        x=[row["pct"]], y=["share"], orientation="h",
        name=row["label"], marker_color=color, marker_line_color="white",
        marker_line_width=2,
        text=f"{row['pct']:.1f}%",
        textposition="inside", insidetextanchor="middle",
        textfont=dict(color="white", size=14),
        hovertemplate=f"<b>{row['label']}</b><br>n = {row['n']:,}<br>%{{x:.2f}}%<extra></extra>",
    ))
    left += row["pct"]
fig.update_layout(
    title="UKATGEORIE — class distribution (full dataset)",
    barmode="stack", height=240, showlegend=True,
    legend=dict(orientation="h", y=-0.4, x=0.5, xanchor="center"),
    xaxis=dict(range=[0, 100], title="share (%)"),
    yaxis=dict(showticklabels=False),
    margin=dict(l=20, r=20, t=60, b=80),
)
save_fig(fig, "04_target_distribution_stacked")
fig.show()


### 4.2  Stability across years

A model trained on 2016 – 2022 will be evaluated on 2024. If class
proportions drift across years, the held-out performance estimate is biased.

In [ ]:
target_by_year = con.execute(f"""
    SELECT UJAHR, UKATGEORIE, COUNT(*) AS n
    FROM '{DATA}'
    GROUP BY UJAHR, UKATGEORIE
    ORDER BY UJAHR, UKATGEORIE
""").df()
pivot = target_by_year.pivot(index="UJAHR", columns="UKATGEORIE", values="n")
pct = pivot.div(pivot.sum(axis=1), axis=0) * 100
pct.columns = ["1 — Getötet", "2 — Schwer", "3 — Leicht"]
pct.round(2)


In [ ]:
pct_long = pct.reset_index().melt(id_vars="UJAHR", var_name="class", value_name="pct")
fig = px.line(
    pct_long, x="UJAHR", y="pct", color="class", markers=True,
    title="Class proportions by year — stability check",
    labels={"UJAHR": "", "pct": "share (%)", "class": "severity"},
    color_discrete_sequence=COLOURS_SEV,
    height=DEFAULT_FIG_H,
)
fig.update_traces(hovertemplate="<b>%{x}</b><br>%{y:.2f}%<extra></extra>")
fig.update_layout(xaxis=dict(tickmode="linear"))
save_fig(fig, "04_target_stability_by_year")
fig.show()


> **Observation.** Class shares are stable to within ~1 pp across the nine
> years — no structural drift that would invalidate the chronological split.

---

## 5 — Univariate distributions

A first look at each feature in isolation. Categorical features get
countplots; binary features a grid of counts; continuous features histograms.

In [ ]:
# Categorical features — 2 × 2 grid of countplots.
cat_cols = ["ULICHTVERH", "STRZUSTAND", "UART", "UTYP1"]
counts = {
    col: con.execute(
        f"SELECT {col} AS v, COUNT(*) AS n FROM '{DATA}' GROUP BY {col} ORDER BY {col}"
    ).df()
    for col in cat_cols
}

fig = make_subplots(rows=2, cols=2, subplot_titles=cat_cols,
                    vertical_spacing=0.18, horizontal_spacing=0.10)
positions = [(1, 1), (1, 2), (2, 1), (2, 2)]
for col, (r, c) in zip(cat_cols, positions):
    d = counts[col]
    fig.add_trace(
        go.Bar(
            x=d["v"].astype(str), y=d["n"],
            marker_color=COLOR_PRIMARY,
            hovertemplate=f"{col} = %{{x}}<br>n = %{{y:,}}<extra></extra>",
            showlegend=False,
        ),
        row=r, col=c,
    )
fig.update_layout(
    title="Categorical feature distributions",
    height=620, margin=dict(t=80),
)
save_fig(fig, "05_categorical_grid")
fig.show()


In [ ]:
# Binary transport-mode flags.
mode_counts = con.execute(f"""
    SELECT 'IstRad'     AS mode, SUM(CAST(IstRad     AS INT)) AS n FROM '{DATA}' UNION ALL
    SELECT 'IstPKW'     AS mode, SUM(CAST(IstPKW     AS INT)) AS n FROM '{DATA}' UNION ALL
    SELECT 'IstFuss'    AS mode, SUM(CAST(IstFuss    AS INT)) AS n FROM '{DATA}' UNION ALL
    SELECT 'IstKrad'    AS mode, SUM(CAST(IstKrad    AS INT)) AS n FROM '{DATA}' UNION ALL
    SELECT 'IstGkfz'    AS mode, SUM(CAST(IstGkfz    AS INT)) AS n FROM '{DATA}' UNION ALL
    SELECT 'IstSonstig' AS mode, SUM(CAST(IstSonstig AS INT)) AS n FROM '{DATA}'
""").df().sort_values("n", ascending=True)

fig = px.bar(
    mode_counts, x="n", y="mode", orientation="h",
    title="Transport-mode involvement — rows with each flag = 1",
    labels={"n": "accidents", "mode": ""},
    color_discrete_sequence=[COLOR_PRIMARY],
    height=380,
)
fig.update_traces(
    hovertemplate="<b>%{y}</b><br>%{x:,} accidents<extra></extra>",
    texttemplate="%{x:,}", textposition="outside",
)
save_fig(fig, "05_transport_mode_counts")
fig.show()


In [ ]:
# Geographic coordinate histograms with skew annotation.
coords = con.execute(f"SELECT LON, LAT FROM '{DATA}'").df()
skew_lon = float(coords['LON'].skew())
skew_lat = float(coords['LAT'].skew())

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=(f"LON — skew = {skew_lon:.2f}", f"LAT — skew = {skew_lat:.2f}"),
    horizontal_spacing=0.10,
)
fig.add_trace(go.Histogram(x=coords["LON"], nbinsx=60, marker_color=COLOR_PRIMARY,
                           name="LON", showlegend=False), row=1, col=1)
fig.add_trace(go.Histogram(x=coords["LAT"], nbinsx=60, marker_color=COLOR_PRIMARY,
                           name="LAT", showlegend=False), row=1, col=2)
fig.update_layout(title="Coordinate distributions", height=380)
save_fig(fig, "05_coords_histograms")
fig.show()


> **Observation.** Coordinate distributions reflect Germany's
> centre-of-mass — neither feature is heavily skewed in a way that demands
> transformation. Tree models will use these directly; if a distance-based
> baseline is added in A³, `StandardScaler` is appropriate.

---

## 6 — Bivariate analysis — feature × target

Pearson correlation on coded categoricals is meaningless. The correct
measure for categorical-vs-categorical association is **Cramér's V**
(range 0 … 1).

In [ ]:
def cramers_v(x: pd.Series, y: pd.Series) -> float:
    """Bias-corrected Cramér's V (Bergsma & Wicher, 2013)."""
    from scipy.stats import chi2_contingency
    confusion = pd.crosstab(x, y)
    n = confusion.values.sum()
    if n == 0:
        return float("nan")
    chi2 = chi2_contingency(confusion, correction=False)[0]
    phi2 = chi2 / n
    r, k = confusion.shape
    phi2c = max(0.0, phi2 - ((k - 1) * (r - 1)) / (n - 1))
    rc = r - ((r - 1) ** 2) / (n - 1)
    kc = k - ((k - 1) ** 2) / (n - 1)
    denom = min(kc - 1, rc - 1)
    return float("nan") if denom <= 0 else np.sqrt(phi2c / denom)

CV_SAMPLE = 100_000
df_cv = con.execute(
    f"SELECT * FROM '{DATA}' USING SAMPLE {CV_SAMPLE} ROWS (reservoir, 42)"
).df()

cv_cols = ["UKATGEORIE", "UART", "UTYP1", "ULICHTVERH", "STRZUSTAND",
           "UWOCHENTAG", "UMONAT", "IstRad", "IstPKW", "IstFuss",
           "IstKrad", "IstGkfz", "IstSonstig"]

cv_matrix = pd.DataFrame(index=cv_cols, columns=cv_cols, dtype=float)
for a in cv_cols:
    for b in cv_cols:
        cv_matrix.loc[a, b] = cramers_v(df_cv[a], df_cv[b])
cv_matrix.round(3)


In [ ]:
fig = px.imshow(
    cv_matrix.astype(float).values,
    x=cv_matrix.columns, y=cv_matrix.index,
    color_continuous_scale="Plasma",
    zmin=0, zmax=1,
    text_auto=".2f", aspect="equal",
    title=f"Cramér's V — categorical association (n = {CV_SAMPLE:,} sample)",
    labels=dict(color="Cramér's V"),
    height=640,
)
fig.update_traces(
    hovertemplate="<b>%{y}</b> × <b>%{x}</b><br>Cramér's V = %{z:.3f}<extra></extra>",
)
fig.update_xaxes(tickangle=-45)
save_fig(fig, "06_cramers_v_heatmap")
fig.show()


> **Reading guide.** The first row / column (`UKATGEORIE`) shows each
> feature's association with the target. Values above ~0.3 indicate a
> meaningful relationship; very high values on features that *should not*
> dictate the target are leakage flags and warrant the conditional-entropy
> probe in §9.

In [ ]:
# Severity-share decomposition by lighting and road condition.
fig = make_subplots(
    rows=1, cols=2, shared_yaxes=True,
    subplot_titles=("Severity share by ULICHTVERH", "Severity share by STRZUSTAND"),
    horizontal_spacing=0.10,
)
labels_sev = ["1 Getötet", "2 Schwer", "3 Leicht"]
for col_i, col in enumerate(["ULICHTVERH", "STRZUSTAND"], start=1):
    pct_df = con.execute(f"""
        SELECT {col} AS x, UKATGEORIE AS sev, COUNT(*) AS n
        FROM '{DATA}' GROUP BY {col}, UKATGEORIE
    """).df()
    pivot = pct_df.pivot(index="x", columns="sev", values="n").fillna(0)
    pivot = pivot.div(pivot.sum(axis=1), axis=0) * 100
    for sev_i, (sev, color, label) in enumerate(zip([1, 2, 3], COLOURS_SEV, labels_sev)):
        if sev not in pivot.columns:
            continue
        fig.add_trace(
            go.Bar(
                x=pivot.index.astype(str), y=pivot[sev],
                name=label, marker_color=color,
                showlegend=(col_i == 1),
                hovertemplate=f"{col} = %{{x}}<br>{label}: %{{y:.2f}}%<extra></extra>",
            ),
            row=1, col=col_i,
        )
fig.update_layout(
    barmode="stack", title="Severity decomposition by environment",
    height=460, legend=dict(orientation="h", y=-0.18, x=0.5, xanchor="center"),
)
fig.update_yaxes(title="share (%)", col=1)
save_fig(fig, "06_severity_by_conditions")
fig.show()


---

## 7 — Temporal patterns

Hourly profile, weekday × hour heatmaps, and stability checks.

In [ ]:
hourly = con.execute(f"""
    SELECT USTUNDE AS hour,
           COUNT(*) AS n,
           AVG(CAST(UKATGEORIE AS DOUBLE)) AS mean_severity
    FROM '{DATA}'
    GROUP BY USTUNDE
    ORDER BY USTUNDE
""").df()

fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(
    go.Bar(
        x=hourly["hour"], y=hourly["n"],
        marker_color=COLOR_PRIMARY, opacity=0.85, name="count",
        hovertemplate="hour %{x}<br>%{y:,} accidents<extra></extra>",
    ),
    secondary_y=False,
)
fig.add_trace(
    go.Scatter(
        x=hourly["hour"], y=hourly["mean_severity"],
        mode="lines+markers", marker=dict(color=COLOR_FATAL, size=8),
        line=dict(color=COLOR_FATAL, width=2),
        name="mean UKATGEORIE (lower = more severe)",
        hovertemplate="hour %{x}<br>mean severity %{y:.3f}<extra></extra>",
    ),
    secondary_y=True,
)
fig.update_layout(
    title="Hourly profile — frequency and mean severity",
    height=DEFAULT_FIG_H,
    legend=dict(orientation="h", y=-0.20, x=0.5, xanchor="center"),
)
fig.update_xaxes(title="hour of day", tickmode="linear", dtick=1)
fig.update_yaxes(title_text="accidents", secondary_y=False)
fig.update_yaxes(title_text="mean UKATGEORIE", secondary_y=True)
save_fig(fig, "07_hourly_profile")
fig.show()


In [ ]:
# Weekday × hour heatmaps — count + mean severity.
wh = con.execute(f"""
    SELECT UWOCHENTAG AS weekday, USTUNDE AS hour,
           COUNT(*) AS n,
           AVG(CAST(UKATGEORIE AS DOUBLE)) AS mean_severity
    FROM '{DATA}'
    GROUP BY UWOCHENTAG, USTUNDE
""").df()

# Weekday coding: 1=Sun, 2=Mon … 7=Sat. Reorder Mon-first.
weekday_names = {1: "Sun", 2: "Mon", 3: "Tue", 4: "Wed", 5: "Thu", 6: "Fri", 7: "Sat"}
order = [2, 3, 4, 5, 6, 7, 1]

count_pivot = wh.pivot(index="weekday", columns="hour", values="n").reindex(order)
sev_pivot   = wh.pivot(index="weekday", columns="hour", values="mean_severity").reindex(order)
y_labels    = [weekday_names[i] for i in order]
x_labels    = list(count_pivot.columns)

fig = make_subplots(
    rows=2, cols=1,
    subplot_titles=("Weekday × hour — accident count",
                    "Weekday × hour — mean UKATGEORIE (darker = more severe)"),
    vertical_spacing=0.18,
)
fig.add_trace(
    go.Heatmap(
        z=count_pivot.values, x=x_labels, y=y_labels,
        colorscale="Viridis",
        colorbar=dict(title="count", y=0.78, len=0.42),
        hovertemplate="%{y} %{x}:00<br>%{z:,} accidents<extra></extra>",
    ),
    row=1, col=1,
)
fig.add_trace(
    go.Heatmap(
        z=sev_pivot.values, x=x_labels, y=y_labels,
        colorscale="Reds_r",
        colorbar=dict(title="mean severity", y=0.22, len=0.42),
        hovertemplate="%{y} %{x}:00<br>mean severity = %{z:.3f}<extra></extra>",
    ),
    row=2, col=1,
)
fig.update_layout(title="Weekday × hour patterns", height=720)
fig.update_xaxes(title_text="hour", row=2, col=1)
save_fig(fig, "07_weekday_hour_heatmaps")
fig.show()


> **Observation.** Frequency peaks during commuter hours (7 – 9 and
> 15 – 18) on weekdays; severity is *higher* (lower mean) at night and on
> weekends — the two plots tell different stories, which is why both must
> be shown. A model that uses only `USTUNDE` sees the count signal; a model
> that uses `USTUNDE` and `UWOCHENTAG` together can resolve the interaction.

---

## 8 — Spatial patterns

Two views: an interactive density map on a representative sample, and a
Bundesland-level aggregate of fatal-accident share. The Bundesland is
derived from the first two digits of `UKREIS` (per `AGENTS.md`).

In [ ]:
# Interactive density map (OpenStreetMap, no token required).
SAMPLE_GEO = 50_000
geo_sample = con.execute(
    f"SELECT LON, LAT, UKATGEORIE FROM '{DATA}' "
    f"USING SAMPLE {SAMPLE_GEO} ROWS (reservoir, 42)"
).df()
geo_sample["severity_label"] = geo_sample["UKATGEORIE"].map({
    1: "1 — Getötet", 2: "2 — Schwer", 3: "3 — Leicht",
})

fig = px.scatter_mapbox(
    geo_sample,
    lat="LAT", lon="LON", color="severity_label",
    color_discrete_map={
        "1 — Getötet": COLOR_FATAL,
        "2 — Schwer":  COLOR_SERIOUS,
        "3 — Leicht":  COLOR_MINOR,
    },
    category_orders={"severity_label": ["1 — Getötet", "2 — Schwer", "3 — Leicht"]},
    opacity=0.45, zoom=5.2,
    center={"lat": 51.2, "lon": 10.4},
    mapbox_style="open-street-map",
    title=f"Accident locations — sample of {SAMPLE_GEO:,} rows",
    height=640,
)
fig.update_traces(marker=dict(size=4))
fig.update_layout(
    legend=dict(orientation="h", y=-0.02, x=0.5, xanchor="center"),
    margin=dict(l=0, r=0, t=60, b=20),
)
save_fig(fig, "08_geo_density_map")
fig.show()


In [ ]:
BL_NAMES = {
    1: "Schleswig-Holstein", 2: "Hamburg", 3: "Niedersachsen", 4: "Bremen",
    5: "Nordrhein-Westfalen", 6: "Hessen", 7: "Rheinland-Pfalz",
    8: "Baden-Württemberg", 9: "Bayern", 10: "Saarland", 11: "Berlin",
    12: "Brandenburg", 13: "Mecklenburg-Vorpommern", 14: "Sachsen",
    15: "Sachsen-Anhalt", 16: "Thüringen",
}

bl = con.execute(f"""
    SELECT CAST(SUBSTR(UKREIS, 1, 2) AS INT) AS uland,
           COUNT(*) AS n,
           AVG(CAST(UKATGEORIE AS DOUBLE)) AS mean_severity,
           SUM(CASE WHEN UKATGEORIE = 1 THEN 1 ELSE 0 END) AS n_fatal,
           100.0 * SUM(CASE WHEN UKATGEORIE = 1 THEN 1 ELSE 0 END)
                 / COUNT(*) AS pct_fatal
    FROM '{DATA}'
    GROUP BY uland
    ORDER BY uland
""").df()
bl["name"] = bl["uland"].map(BL_NAMES)
bl


In [ ]:
bl_sorted = bl.sort_values("pct_fatal", ascending=True)
fig = px.bar(
    bl_sorted, x="pct_fatal", y="name", orientation="h",
    title="Share of fatal accidents (class 1) by Bundesland",
    labels={"pct_fatal": "% fatal", "name": ""},
    color_discrete_sequence=[COLOR_FATAL],
    height=560,
    custom_data=["n", "n_fatal"],
)
fig.update_traces(
    hovertemplate=(
        "<b>%{y}</b><br>"
        "%{x:.2f}% fatal<br>"
        "total = %{customdata[0]:,}<br>"
        "fatal = %{customdata[1]:,}<extra></extra>"
    ),
    texttemplate="%{x:.2f}%", textposition="outside",
)
save_fig(fig, "08_pct_fatal_by_bundesland")
fig.show()


> **Observation.** Rural Bundesländer (Mecklenburg-Vorpommern, Brandenburg,
> Sachsen-Anhalt) show systematically higher shares of fatal accidents than
> dense urban ones (Hamburg, Berlin, Bremen). This is consistent with the
> literature: higher speeds, longer rescue times, lower infrastructure
> density. The Q-phase scoping note about "no causal claims" applies — the
> model can use this signal but the project does not assert that moving to
> the city makes one safer.

---

## 9 — Leakage audit

Three checks: target leakage on suspect features, temporal leakage via the
chronological split, and physical no-overlap between splits.

### 9.1  Target-leakage probe

`UART` (Unfallart) and `UTYP1` (Unfalltyp) describe *what kind of accident*
happened. They are recorded after the event and may partially encode the
severity outcome. We measure the conditional-entropy reduction:

$$\text{reduction} = 1 - \frac{H(\text{UKATGEORIE} \mid X)}{H(\text{UKATGEORIE})}$$

A reduction near 100 % means the feature definitionally encodes the target.

In [ ]:
def entropy(values: pd.Series) -> float:
    counts = Counter(values.dropna())
    total = sum(counts.values())
    return -sum((c / total) * log2(c / total) for c in counts.values() if c > 0)

def conditional_entropy(y: pd.Series, x: pd.Series) -> float:
    """H(Y | X) in bits."""
    pair = pd.DataFrame({"x": x, "y": y}).dropna()
    if len(pair) == 0:
        return float("nan")
    n = len(pair)
    h = 0.0
    for _, grp in pair.groupby("x", observed=True):
        p_x = len(grp) / n
        h += p_x * entropy(grp["y"])
    return h

probe = con.execute(
    f"SELECT UKATGEORIE, UART, UTYP1, ULICHTVERH, STRZUSTAND, USTUNDE "
    f"FROM '{DATA}' USING SAMPLE 200000 ROWS (reservoir, 42)"
).df()

H_y = entropy(probe["UKATGEORIE"])
print(f"H(UKATGEORIE) = {H_y:.4f} bits  (marginal entropy of the target)\n")
print(f"{'feature':<12s} {'H(Y|X)':>10s} {'reduction':>12s}  verdict")
print("-" * 55)
probe_results = []
for col in ["UART", "UTYP1", "ULICHTVERH", "STRZUSTAND", "USTUNDE"]:
    H_y_given_x = conditional_entropy(probe["UKATGEORIE"], probe[col])
    reduction = 1 - H_y_given_x / H_y
    flag = "→ LEAKAGE RISK" if reduction > 0.5 else "ok"
    print(f"{col:<12s} {H_y_given_x:>10.4f} {reduction:>11.1%}  {flag}")
    probe_results.append({"feature": col, "H_cond": H_y_given_x,
                          "reduction": reduction, "risk": reduction > 0.5})
probe_df = pd.DataFrame(probe_results)


In [ ]:
# Visualise the entropy reduction per feature.
probe_df_sorted = probe_df.sort_values("reduction", ascending=True)
probe_df_sorted["color"] = probe_df_sorted["risk"].map({True: COLOR_FATAL, False: COLOR_PRIMARY})
fig = go.Figure()
fig.add_trace(go.Bar(
    x=probe_df_sorted["reduction"] * 100,
    y=probe_df_sorted["feature"],
    orientation="h",
    marker_color=probe_df_sorted["color"].tolist(),
    text=[f"{r:.1%}" for r in probe_df_sorted["reduction"]],
    textposition="outside",
    hovertemplate="<b>%{y}</b><br>entropy reduction = %{x:.2f}%<extra></extra>",
))
fig.add_vline(x=50, line_dash="dash", line_color="grey",
              annotation_text="50 % threshold", annotation_position="top")
fig.update_layout(
    title="Conditional-entropy reduction of UKATGEORIE given each feature",
    xaxis_title="reduction (%)", yaxis_title="",
    height=380, margin=dict(l=80, r=80, t=80, b=40),
    showlegend=False,
)
save_fig(fig, "09_leakage_probe_bars")
fig.show()


> **Decision rule.** A reduction > 50 % triggers a definitional review of
> the feature — does it encode information not available at the time of the
> police report? If yes, the feature is excluded in A³. If no (the
> association is genuine domain signal), the feature is retained but the
> finding is documented in the §10 decision table.

### 9.2  Chronological split verification

In [ ]:
split = con.execute(f"""
    SELECT
        CASE WHEN UJAHR <= 2022 THEN 'train (2016–2022)'
             WHEN UJAHR  = 2023 THEN 'val   (2023)'
             ELSE               'test  (2024)' END AS split,
        MIN(UJAHR) AS year_min,
        MAX(UJAHR) AS year_max,
        COUNT(*) AS n,
        ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct
    FROM '{DATA}'
    GROUP BY split
    ORDER BY year_min
""").df()
split


In [ ]:
split_target = con.execute(f"""
    SELECT
        CASE WHEN UJAHR <= 2022 THEN 'train'
             WHEN UJAHR  = 2023 THEN 'val'
             ELSE               'test'  END AS split,
        UKATGEORIE, COUNT(*) AS n
    FROM '{DATA}'
    GROUP BY split, UKATGEORIE
""").df()
pivot = split_target.pivot(index="split", columns="UKATGEORIE", values="n")
pct   = pivot.div(pivot.sum(axis=1), axis=0) * 100
pct.columns = ["1 Getötet", "2 Schwer", "3 Leicht"]
pct.reindex(["train", "val", "test"]).round(2)


### 9.3  No-overlap check

A sanity check: no `OBJECTID` may appear in more than one split.

In [ ]:
overlap = con.execute(f"""
    WITH labelled AS (
        SELECT OBJECTID,
               CASE WHEN UJAHR <= 2022 THEN 'train'
                    WHEN UJAHR  = 2023 THEN 'val'
                    ELSE               'test'  END AS split
        FROM '{DATA}'
    )
    SELECT OBJECTID, COUNT(DISTINCT split) AS n_splits
    FROM labelled
    GROUP BY OBJECTID
    HAVING COUNT(DISTINCT split) > 1
""").df()
print("OBJECTIDs appearing in more than one split:", len(overlap))


> **Verdict.** The chronological split is well-formed, class distribution is
> stable across splits, and no OBJECTID overlap exists. Temporal leakage is
> structurally prevented.

---

## 10 — Preprocessing decisions (U → A³ handover)

The following table specifies, per column, what A³ must do. The U phase
*decides*; A³ *implements*, inside a `Pipeline` so preprocessing statistics
are fit on training data only.

| Column | Missing strategy | Encoding | Scaling | Notes |
|:---|:---|:---|:---|:---|
| `OBJECTID` | n/a | drop before fit | n/a | identifier only; never a feature |
| `UJAHR` | n/a | drop before fit | n/a | used to define the split, not as a feature |
| `UMONAT` | drop row if missing | sin/cos cyclic (period 12) | n/a | seasonality observed |
| `USTUNDE` | drop row if missing | sin/cos cyclic (period 24) | n/a | strong daily structure |
| `UWOCHENTAG` | drop row if missing | sin/cos cyclic (period 7) | n/a | weekly structure observed |
| **`UKATGEORIE`** | drop row if missing | label — no encoding | n/a | target |
| `UART` | mode | one-hot or target-encoded | n/a | **§9.1 probe result must be acceptable before inclusion** |
| `UTYP1` | mode | one-hot or target-encoded | n/a | same as `UART` |
| `ULICHTVERH` | mode | one-hot | n/a | 3 nominal levels |
| `STRZUSTAND` | mode | one-hot | n/a | 3 nominal levels |
| `IstRad … IstSonstig` | none observed | pass-through | n/a | binary |
| `LON`, `LAT` | drop row if outside DE bbox | none for tree models | `StandardScaler` *only if* a distance-based baseline is added | both retained |
| `UREGBEZ` | mode | target-encoding with smoothing | n/a | moderate cardinality |
| `UKREIS` | mode | target-encoding with smoothing | n/a | high cardinality (~400 levels) |
| `UGEMEINDE` | drop column | n/a | n/a | very-high cardinality, no benefit over `UKREIS` |

### Imbalance handling

A class imbalance of ≈ 1 : 18 : 81 is observed. The Q phase chose macro-F1
and recall-on-class-1 as the metrics that protect against majority-class
collapse. **A³ chooses the mitigation**, from this menu, and reports the
selection: `class_weight="balanced"`, SMOTE / ADASYN, threshold moving, or
ordinal classification. The U phase does not pre-commit.

### Cross-validation hint

Time-series semantics. Within the 2016 – 2022 training window, A³ should
use either a chronological `TimeSeriesSplit` or a year-grouped K-fold;
*not* a random `StratifiedKFold` that would let the model "see the future"
inside CV.

---

## 11 — Summary

### Dataset characterisation

- **Volume:** ~2.09 M rows · 21 columns · 9 vintages 2016 – 2024.
- **Quality:** no duplicate OBJECTIDs; row-duplicate count negligible;
  geographic outliers handled by explicit bounding box; missingness
  documented per column with imputation strategy.
- **Target:** class imbalance ≈ 1 % / 18 % / 81 %, stable across years and
  splits.
- **Splits:** chronological, train 2016 – 2022 / val 2023 / test 2024;
  no OBJECTID overlap; class proportions stable across splits.
- **Patterns:** bimodal hourly distribution (commuter and afternoon peaks);
  severity inverts the count signal (more severe at night and weekends);
  rural Bundesländer carry higher shares of fatal outcomes.
- **Leakage:** conditional-entropy probe on `UART` / `UTYP1` reported in
  §9.1; treat as definitional features only if reduction ≤ 50 %.

### Top-3 risks for A³

1. **Definitional leakage from `UART` / `UTYP1`.** If the §9.1 probe reports
   conditional-entropy reduction > 50 %, these features encode the target
   and must be excluded — high training scores would not transfer to
   deployment.
2. **Imbalance collapse on macro-F1.** Without class weights or sampling,
   tree models default to majority-class prediction on minority instances;
   recall on class 1 will fall below the 0.50 acceptance threshold.
3. **Stationarity assumption between 2016 – 2022 and 2024.** COVID-19
   produced a structural year (2020). If A³ trains naively, the model
   learns the COVID-year distribution as if it were normal; consider a
   year-weight or drop 2020 from training and document the choice.

### U-phase acceptance checklist

```text
[ ] Provenance block at top — versions, hash, git commit, seed
[ ] Schema printed and annotated with semantic types
[ ] Cardinality + missingness table rendered
[ ] Missingness map on a sample rendered
[ ] Sentinel-value scan executed
[ ] Duplicate detection (OBJECTID + exact rows)
[ ] Range / domain bound checks on coordinates and ordinals
[ ] Consistency rule check (at least one transport mode set)
[ ] Target distribution + imbalance ratio computed
[ ] Target stability across years verified
[ ] Univariate countplots / histograms for all relevant features
[ ] Cramér's V matrix rendered
[ ] Conditional severity plots for two key features
[ ] Hourly profile + weekday × hour heatmaps
[ ] Geographic density map + Bundesland aggregate
[ ] Conditional-entropy leakage probe executed for UART, UTYP1
[ ] Chronological split sizes verified
[ ] Class stability across splits verified
[ ] No OBJECTID overlap between splits
[ ] §10 preprocessing decision table filled per column
[ ] Top-3 risks for A³ written
[ ] All plots exported to reports/figures/u_phase/
[ ] Notebook runs end-to-end without manual intervention
```

> **Transition.** The dataset is audited, the leakage probes are run, and
> the preprocessing contract is written. Proceed to `03_A3_Phase.ipynb` to
> implement the decisions above and train the first baseline models.
